In [26]:
##########################################################################################################
#Fit for m, theta, chi_i .....
#all the other data is given... 
##########################################################################################################

In [3]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad
from matplotlib.ticker import MultipleLocator, FuncFormatter

In [9]:
pulsars = np.array(['J0030+0451', 'J0613−0200', 'J0751+1807', 'J0900−3144', 
                    'J1012+5307', 'J1022+1001', 'J1024−0719', 'J1455−3330',
                    'J1600−3053', 'J1640+2224', 'J1713+0747', 'J1730−2304',
                    'J1738+0333', 'J1744−1134', 'J1751−2857', 'J1801−1417',
                    'J1804−2717', 'J1843−1113', 'J1857+0943', 'J1909−3744',
                    'J1910+1256', 'J1911+1347', 'J1918−0642', 'J2124−3358',
                    'J2322+2057'])

In [3]:
#given parameters 
G = 6.7e-39 #GeV^-2
rho = 2.3e-42  #GeV^4
delta = 0.1
H = 1.5e-42 #GeV 
a = 1.0
deltadot = H*delta /(2*np.sqrt(np.pi))

#given by measurement
#table B.1-B.7 
nu = np.array([205.530695938456, 326.6005620234831, 287.457853995106, 90.011841919354, 
               190.2678344415654, 60.7794479566973, 193.715683448548, 125.200243244993, 
               277.9377069896062, 316.123979331869, 218.8118404171605, 123.110287147370,
               170.937369887100, 245.4261196898081, 255.43611088568, 275.85470899694,
               107.031649219533, 541.809745036152, 186.4940783779890, 339.3156872184705,
               200.658802230113, 216.171227371979, 130.789514123371, 202.793893746013,
               207.96816335836]) #Hz
#table 2: 
#mes_time
tend = np.array([22.0, 22.9, 24.2, 13.6, 23.7, 24.5, 23.1, 15.7, 14.3, 24.4, 24.5, 16.1,
                 14.1, 24.0, 14.7, 13.7, 14.7, 16.8, 24.1, 15.7, 15.2, 14.2, 19.7, 16.0, 14.7]) #yr
#whitened rms residual
wrms = np.array([2.30, 1.06, 1.50, 2.60, 1.02, 1.56, 1.02, 2.46, 0.37, 1.10, 0.20, 0.83,
                 2.33, 0.56, 2.34, 2.46, 1.63, 0.81, 1.05, 0.14, 1.77, 0.75, 1.31, 2.17, 4.08]) #mikrosec 




In [ ]:
#priors for m, theta, chi_i 

In [6]:
#usefull combinations
Theta = lambda t, theta, m: m*t + theta
preX = lambda m: 4*np.pi*G*rho*delta/m**3
preY = lambda k, m: 0 #k->inf

#deltaT
def XY(time, f, theta, m):  
    t1, t2, t3, t4 = time+a*chi, time, time+a*chi+T(time), time+T(time)
    return f(2*Theta(t1, theta, m)) - f(2*Theta(t2, theta, m)) - f(2*Theta(t3, theta, m)) - f(2*Theta(t4, theta, m)) 

def deltaT(time, theta, k, m): 
    return preX(m) * XY(time, np.sin, theta, m) + preY(k, m)* XY(time, np.cos, theta, m)

#Residual
def R(tend, theta, k, m):
    innerfct = lambda t: deltaT(t, theta, k, m)/T(t)
    tend = tend % (np.pi/m) #integration over full period = 0 -> neglegtion before integration is less expensive
    return quad(innerfct, 0.0, tend)[0]

In [8]:
#Plotting
p_read = 1.5 #factor for fontsize of axes, ticks, legend

def plot_reflines(axes, refline, color, label):
    axes.axvline(refline, linestyle=':', color=color, label=label)
    
def plot_residual(axes, x, y, label=None, refline='None', reflinelabel=None, 
                  xscale='linear', yscale='linear', xlabel=None, ylabel=None):
    axes.plot(x, y, color='black', linestyle='None', marker='d', markeredgecolor='black', markersize=3, label=label)
    if refline!=None: plot_reflines(axes, refline, color='green', label=reflinelabel) #this ref-value is used in other plots
    axes.set_xscale(xscale)
    axes.set_yscale(yscale)
    axes.tick_params(axis='both', labelsize=8*p_read)
    plt.grid(axis='y', color='gainsboro') 
    plt.xlabel(xlabel, fontsize=10*p_read)
    plt.ylabel(ylabel, fontsize=10*p_read)
    plt.tight_layout()